In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

BASE = Path(r"C:\Users\matas\Desktop\LEI\Data")
OUT = BASE / "outputs"

CSV_HILDA  = OUT / "hilda_lithuania_timeseries.csv"
CSV_LUCAS  = OUT / "lucas_lithuania_timeseries.csv"
CSV_HYDE   = OUT / "hyde_lithuania_timeseries.csv"
CSV_LUH2   = OUT / "luh2_lithuania_timeseries.csv"
CSV_CORINE = OUT / "corine_lithuania_timeseries.csv"

In [2]:
def load_timeseries(csv_path, dataset_name):
    df = pd.read_csv(csv_path)
    df["dataset"] = dataset_name
    return df

hilda  = load_timeseries(CSV_HILDA, "hilda")
lucas  = load_timeseries(CSV_LUCAS, "lucas")
hyde   = load_timeseries(CSV_HYDE, "hyde")
luh2   = load_timeseries(CSV_LUH2, "luh2")
corine = load_timeseries(CSV_CORINE, "corine")

hilda.head(), corine.head()

(   year  class_id   class_name  count dataset
 0  1910         1        Water     64   hilda
 1  1910         3        Urban    352   hilda
 2  1910         4  Agriculture  84201   hilda
 3  1910         5       Forest   5669   hilda
 4  1911         1        Water     64   hilda,
    year  class_id   class_name   count dataset
 0  1990         1        Water    6891  corine
 1  1990         2     Wetlands    3192  corine
 2  1990         3        Urban   11972  corine
 3  1990         4  Agriculture  224459  corine
 4  1990         5       Forest  116801  corine)

In [3]:
def to_shares(df):
    # df: year, class_id, class_name, count, dataset
    g = df.groupby(["dataset", "year", "class_name"], as_index=False)["count"].sum()
    totals = g.groupby(["dataset", "year"])["count"].transform("sum")
    g["share"] = g["count"] / totals
    # pivot to wide: columns = class_name_share
    wide = g.pivot_table(
        index=["dataset", "year"],
        columns="class_name",
        values="share",
        fill_value=0.0,
    )
    wide.columns = [f"share_{c}" for c in wide.columns]
    return wide.reset_index()

ts_hilda  = to_shares(hilda)
ts_lucas  = to_shares(lucas)
ts_hyde   = to_shares(hyde)
ts_luh2   = to_shares(luh2)
ts_corine = to_shares(corine)

ts_hilda.head()

,dataset,year,share_Agriculture,share_Forest,share_Urban,share_Water
0,hilda,1910,0.932603,0.062789,0.003899,0.000709
1,hilda,1911,0.933279,0.062103,0.003910,0.000709
2,hilda,1912,0.933866,0.061527,0.003899,0.000709
3,hilda,1913,0.935051,0.060331,0.003910,0.000709
4,hilda,1914,0.935627,0.059755,0.003910,0.000709


In [4]:
# Rename CORINE columns to mark them as targets
target = ts_corine.rename(columns=lambda c: c.replace("share_", "corine_") if c.startswith("share_") else c)

# Join predictors by year (inner join)
base = target[["year"] + [c for c in target.columns if c.startswith("corine_")]]

def join_predictor(ts, prefix):
    x = ts.copy()
    x = x.rename(columns=lambda c: c.replace("share_", f"{prefix}_") if c.startswith("share_") else c)
    return base.merge(x.drop(columns=["dataset"]), on="year", how="inner")

# Example: use HILDA + HYDE + LUH2 shares as predictors
joined = base.copy()
for ts, prefix in [(ts_hilda, "hilda"), (ts_hyde, "hyde"), (ts_luh2, "luh2")]:
    x = ts.copy()
    x = x.rename(columns=lambda c: c.replace("share_", f"{prefix}_") if c.startswith("share_") else c)
    joined = joined.merge(x.drop(columns=["dataset"]), on="year", how="inner")

joined.head()


,year,corine_Agriculture,corine_Forest,corine_Urban,corine_Water,corine_Wetlands,hilda_Agriculture,hilda_Forest,hilda_Urban,hilda_Water,hyde_Agriculture,hyde_Forest,hyde_Urban,luh2_Agriculture,luh2_Forest
0,1990,0.617808,0.321487,0.032952,0.018967,0.008786,0.933378,0.059046,0.006867,0.000709,0.938281,0.061719,0.000000,0.70303,0.29697
1,2000,0.615620,0.323626,0.032878,0.018986,0.008890,0.899796,0.090291,0.009204,0.000709,0.938281,0.060250,0.001470,0.70303,0.29697
2,2006,0.605318,0.334002,0.032327,0.019105,0.009248,0.895399,0.092905,0.010987,0.000709,0.938281,0.060250,0.001470,0.70303,0.29697
3,2012,0.589513,0.348397,0.033819,0.019556,0.008714,0.788882,0.197594,0.012815,0.000709,0.937546,0.060250,0.002204,0.70303,0.29697


In [5]:
def run_regression_for_class(class_name: str):
    """Fit RF / XGB / LGBM / CatBoost regressors to predict CORINE class share.

    Target: continuous CORINE share for the given class (0–1).
    Features: all HILDA / HYDE / LUH2 share columns.
    """
    target_col = f"corine_{class_name}"
    assert target_col in joined.columns, f"Missing {target_col} in joined table"

    y = joined[target_col].values

    feature_cols = [c for c in joined.columns if c.startswith(("hilda_", "hyde_", "luh2_"))]
    X = joined[feature_cols].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )

    models = {
        "rf": RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1),
        "xgb": XGBRegressor(
            n_estimators=600,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            n_jobs=-1,
        ),
        "lgbm": LGBMRegressor(
            n_estimators=600,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
        ),
        "cat": CatBoostRegressor(
            iterations=600,
            depth=4,
            learning_rate=0.05,
            loss_function="RMSE",
            verbose=False,
        ),
    }

    results = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        r2 = r2_score(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = float(np.sqrt(mse))
        print(f"=== {class_name} – {name} ===")
        print(f"R²   = {r2:.3f}")
        print(f"RMSE = {rmse:.4f}\n")
        results[name] = {"model": model, "r2": r2, "rmse": rmse}
    if "rf" in results:
        rf = results["rf"]["model"]
        importances = (
            pd.Series(rf.feature_importances_, index=feature_cols)
            .sort_values(ascending=False)
        )
    else:
        importances = pd.Series(dtype="float64")

    return results, importances

# Example usage (run this in a separate cell when ready):
# results_forest, importances_forest = run_regression_for_class("Forest")
# importances_forest.head(20)

In [6]:
results_forest, importances_forest = run_regression_for_class("Forest")
importances_forest.head(20)

=== Forest – rf ===
R²   = -0.194
RMSE = 0.0135

=== Forest – xgb ===
R²   = 0.045
RMSE = 0.0121

[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 2, number of used features: 0
[LightGBM] [Info] Start training from score 0.327745
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requi

C:\Users\matas\miniconda3\envs\landcover2\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\matas\miniconda3\envs\landcover2\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\matas\miniconda3\envs\landcover2\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\matas\miniconda3\envs\landcover2\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs)

=== Forest – cat ===
R²   = -0.026
RMSE = 0.0125



hilda_Forest         0.253968
hilda_Agriculture    0.222222
hilda_Urban          0.201058
hyde_Forest          0.174603
hyde_Urban           0.148148
hyde_Agriculture     0.000000
hilda_Water          0.000000
luh2_Agriculture     0.000000
luh2_Forest          0.000000
dtype: float64